In [29]:
import scipy.io as sio                     # import scipy.io for .mat file I/O 
import numpy as np                         # import numpy
import matplotlib.pyplot as plt            # import matplotlib.pyplot for figure plotting
import torch
import time
from torch.autograd import Variable
import time

In [22]:
def get_mu(lamda,bup_users,bdown_users,bdown2_users,Pmax):
    Rmu = 100
    Lmu = 0
    
    btem = (1+lamda)*bup_users/(bdown_users+lamda*bdown2_users)
    Pcomp = np.sum(np.square(btem),keepdims=True)

    if(Pcomp < Pmax):
        return Lmu
    
    while(Rmu-Lmu > 1e-1):
        midmu = (Rmu + Lmu)/2
        btem = (1+lamda)*bup_users/(bdown_users+lamda*bdown2_users+midmu)
        Pcomp = np.sum(np.square(btem),keepdims=True)
        if(Pcomp < Pmax):
            Rmu = midmu
        else: 
            Lmu = midmu


    return Lmu

In [23]:
def get_lamda(lamda,w_users,Rmin,error):

    sum_w = np.sum(w_users*error-np.log(w_users))

    lamda1 = max(0,lamda+0.01*(sum_w+Rmin-Channel))

    return lamda1

In [24]:
def create_real(num_H, dis, K,Channel):
    CH = np.zeros([num_H, K*Channel, K*Channel]) + 1j*np.zeros([num_H, K*Channel, K*Channel])
    #CH2 = np.zeros([num_H, K, K]) + 1j*np.zeros([num_H, K, K])
    pos = np.zeros([num_H,K*Channel,2])
    distance = np.zeros([num_H, K*Channel, K*Channel])
    Tx = np.arange(K)
    Rx = np.arange(2*K)[K:]

    for i in range(num_H):
        num_squares_per_row =int(np.sqrt(K))  # Number of squares per row and column
        square_size = 50          # Size of each square
        x_ist = np.linspace(square_size / 2, num_squares_per_row * square_size - square_size / 2, num_squares_per_row)
        y_list = np.linspace(square_size / 2, num_squares_per_row * square_size - square_size / 2, num_squares_per_row)
        xv, yv = np.meshgrid(x_ist, y_list)
        xv = xv.flatten()
        yv = yv.flatten()
        locations = list(zip(xv, yv))
        #print(locations)
        for j in range(len(Tx)):
            # Generate points at the center of each square
            pos[i,Tx[j],0] = locations[j][0]
            pos[i,Tx[j],1] = locations[j][1]

            x = pos[i,Tx[j],0]
            y = pos[i,Tx[j],1]
            dice = np.random.uniform(low= dis[0], high = dis[1])
            rad = np.random.uniform(low=0.0, high= 2* np.pi)
            pos[i,Rx[j],:] = [x+dice*np.cos(rad), y + dice*np.sin(rad)] #assume the number of transmitter and receiver are same

        for j in range(K):
            for k in range(K,2*K):
                d = np.linalg.norm(pos[i,j,:] - pos[i,k,:])

                L2 = np.sqrt(min(1,1/(d**2)))
                for m in range(Channel):
                    distance[i,j+m*K,k-K+m*K] = d  #record the distance 
                    
                    CH[i,j+m*K,k-K+m*K]= L2 * 1/np.sqrt(2)*(np.random.randn()+1j*np.random.randn())


                
    return pos, CH, distance

In [25]:
def batch_WMMSE2(p_int, alpha, H, Pmax, var_noise,Channel):
    N = p_int.shape[0]
    K = p_int.shape[1]
    U = int(K/Channel)
    vnew = 0
    b = np.sqrt(p_int)
    f = np.zeros((N,K,1) )
    w = np.zeros( (N,K,1) )
    mu = np.zeros( (N,K,1) )
    lamda = np.zeros( (N,K,1) )
    
    mask = np.eye(K)
    rx_power = np.multiply(H, b)
    rx_power_s = np.square(rx_power)
    valid_rx_power = np.sum(np.multiply(rx_power, mask), 1)
    
    interference = np.sum(rx_power_s, 2) + var_noise
    f = np.divide(valid_rx_power,interference)
    w = 1/(1-np.multiply(f,valid_rx_power))
    #vnew = np.sum(np.log2(w),1)
    bup_users = np.zeros((U,N,Channel))
    bdown_users = np.zeros((U,N,Channel))
    bdown2_users = np.zeros((U,N,Channel))
    power_users = np.zeros((U,N,Channel))
    w_users = np.zeros((U,N,Channel))
    error_users = np.zeros((U,N,Channel))
    
    for ii in range(100):
        #update vk, b is vk in the paper
        fp = np.expand_dims(f,1)
        
        #bp = np.expand_dims(b,1)
        #uk*|h|
        rx_power_uh = np.multiply(H.transpose(0,2,1), fp)#UH
        
        #calculate e
        rx_power_uhv = np.multiply(rx_power_uh, b)#UHV
        valid_rx_power_uhv = np.sum(np.multiply(rx_power_uhv, mask), 1) #UHV valid
        valid_rx_power_uhv_s = np.square(valid_rx_power_uhv-1)
        interference_rx_power_uhv = np.sum(np.multiply(rx_power_uhv, 1-mask), 1) #UHV interference
        interference_rx_power_uhv_s =np.square(interference_rx_power_uhv)
        f_s = np.square(f)
        noise_uhv_s =np.multiply(var_noise,  f_s)
        error_i_m = valid_rx_power_uhv_s+interference_rx_power_uhv_s+noise_uhv_s        
        valid_rx_power_uh = np.sum(np.multiply(rx_power_uh, mask), 1)   #UH
        #alpha*wk*uk*|h|
        
        
        bup = np.multiply(alpha,np.multiply(w,valid_rx_power_uh))     #WUH
        
        valid_rx_power_s_uh = np.square(valid_rx_power_uh)  #(UH)^2
       
        
        rx_power_s_uh = np.square(rx_power_uh)
        wp = np.expand_dims(w,1)
        alphap = np.expand_dims(alpha,1)
        #alpha*wk*uk^2*|h|^2
        bdown = np.sum(np.multiply(alphap,np.multiply(rx_power_s_uh,wp)),2)   #W(UH)^2
        bdown2 =np.multiply(valid_rx_power_s_uh,w)
        
        wpp=np.reshape(wp,(-1, K, 1))

        #mu = np.zeros((U,N,1))
        power_users_modifilied = np.zeros((U,N,Channel))
        normalised_users =[]
        for i in range(U):
            bup_users[i,:,:] = bup[:,i::U]
            bdown_users[i,:,:] = bdown[:,i::U]
            bdown2_users[i,:,:] = bdown2[:,i::U]
            w_users[i,:,:] = np.squeeze(wpp[:,i::U])
            error_users[i,:,:] = error_i_m[:,i::U]
            for j in range(N):
                lamda[j,i,:] = get_lamda(lamda[j,i,:],w_users[i,j,:],Rmin,error_users[i,j,:])
                
                mu[j,i,:] = get_mu(lamda[j,i,:],bup_users[i,j,:],bdown_users[i,j,:],bdown2_users[i,j,:],Pmax)

                
                power_users[i,j,:] = (1+lamda[j,i,:])*bup_users[i,j,:]/(bdown_users[i,j,:]+mu[j,i,:]+lamda[j,i,:]*bdown2_users[i,j,:])
            
            
            b1_normalized = power_users[i,:,:]
            b1 = b1_normalized[:,:,np.newaxis]
            normalised_users.append(b1)
        
        combined=np.concatenate(normalised_users, axis=2)  
        b=np.reshape(combined,(-1, K, 1))
        
        #update uk, f is uk in the paper
        rx_power = np.multiply(H, b)
        rx_power_s = np.square(rx_power)
        valid_rx_power = np.sum(np.multiply(rx_power, mask), 1)
        interference = np.sum(rx_power_s, 2) + var_noise        
        f = np.divide(valid_rx_power,interference)
        #update wk
        w = 1/(1-np.multiply(f,valid_rx_power))
    p_opt = np.square(b)
    return p_opt

In [26]:
def generate_wGaussian(K, num_H, var_noise=1, Pmin=0, seed=2017,Channel=1):
    print('Generate Data ... (seed = %d)' % seed)
    np.random.seed(seed)
    Pmax = 1
    Pini = Pmax*np.ones((num_H,K*Channel,1) )
    alpha = np.ones((num_H,K*Channel))
    fake_a = np.ones((num_H,K*Channel))
    Y=np.zeros((K*Channel,num_H))
    total_time = 0.0
    pos,CH, distance = create_real(num_H, [2,10], K,Channel)
    H=abs(CH)
    start = time.time()
    Y = batch_WMMSE2(Pini,alpha,H,Pmax,var_noise,Channel)
    end = time.time()
    print('wmmse time',end-start)
    Y2=Y
    return H, Y, alpha, Y2, distance

In [27]:
def np_sum_rate(H,p,alpha,var_noise):
    H = np.expand_dims(H,axis=-1)
    K = H.shape[1]
    N = H.shape[-1]
    
    p = p.reshape((-1,K,1,N))
    n= num_test
    abs_H_2 = np.multiply(H, H)
    rx_power = np.multiply(abs_H_2, p)
    rx_power = np.sum(rx_power,axis=-1)
    mask = np.eye(K)
    valid_rx_power = np.sum(np.multiply(rx_power, mask), axis=1)
    interference = np.sum(np.multiply(rx_power, 1 - mask), axis=1) + var_noise
    rate = np.log(1 + np.divide(valid_rx_power, interference))
    w_rate = np.multiply(alpha,rate)
    U = int(K/Channel)
    rate_users = np.zeros((U,n,Channel))

    rate_users_sum_channel  = np.zeros((U,n,1))
    rate_users_sum_channel_all =[]

    for i in range(U):
        rate_users[i,:,:] =w_rate[:,i::U]
        rate_clone = np.copy(rate_users[i,:,:])
        rate_users_sum_channel[i,:,:] = Rmin - np.sum(rate_clone,keepdims=True, axis=1)
        rate_users_sum_channel_all.append(rate_users_sum_channel[i,:,:])
        
    combined_channel=np.concatenate(rate_users_sum_channel_all, axis=1)
    
    sum_positive_rows = 0
    for i in range(n):
        
        if np.all(combined_channel[i] <= 0):  # Check if all elements in the row are larger than 0
            
            sum_positive_rows=1+sum_positive_rows
    print(sum_positive_rows)
    
    Qos_satisfied = (combined_channel <= 0).sum()
    
    sum_rate = np.mean(np.sum(w_rate, axis=1))
    return sum_rate,Qos_satisfied

In [28]:
K =9         # number of users
#num_H = 10000        # number of training samples
Channel = 4
Rmin =3
num_test =10         # number of testing  samples
training_epochs = 50      # number of training epochs
trainseed = 0              # set random seed for training set
testseed = 7               # set random seed for test set
var=0.001
#Xtrain, Ytrain, Atrain, wtime,DisTrain = generate_wGaussian(K, num_H, seed=trainseed, var_noise = var)
X, Y, A, Y2,DisTest = generate_wGaussian(K, num_test, seed=testseed, var_noise = var,Channel=Channel)
sum_rate,Qos_satisfied = np_sum_rate(X.transpose(0,2,1),Y,A,var)
print('wmmse:',sum_rate)
print('QOS:',Qos_satisfied/K/num_test)

Generate Data ... (seed = 7)
wmmse time 0.43772196769714355
6
wmmse: 62.13331591551879
QOS: 0.9555555555555555
